In [2]:
# Step-1 : Building a Custom Decision Tree with Information Gain:
import numpy as np

class CustomDecisionTree:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        self.tree = self._build_tree(X, y)
    
    def _build_tree(self, X, y, depth=0):
        num_samples, num_features = X.shape
        unique_classes = np.unique(y)

        if len(unique_classes) == 1:
            return {'class' : unique_classes[0]}
        if num_samples == 0 or (self.max_depth and depth >= self.max_depth) :
            return {'class' : np.bincount(y).argmax()}

        best_info_gain = -float('inf')
        best_split = None
        for feature_idx in range(num_features):
            thresholds = np.unique(X[:, feature_idx])
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = ~left_mask
                left_y = y[left_mask]
                right_y = y[right_mask]

                info_gain = self._information_gain(y, left_y, right_y)

                if info_gain > best_info_gain:
                    best_info_gain = info_gain
                    best_split = {
                        'feature_idx' : feature_idx,
                        'threshold' : threshold,
                        'left_y' : left_y,
                        'right_y' : right_y,
                    }

        if best_split is None:
            return {'class' : np.bincount(y).argmax()}

        left_tree = self._build_tree(X[best_split['left_y']], best_split['left_y'], depth + 1)
        right_tree = self._build_tree(X[best_split['right_y']], best_split['right_y'], depth + 1)
        return {'feature_idx' : best_split['feature_idx'], 'threshold': best_split['threshold'],
               'left_tree' : left_tree, 'right_tree' : right_tree}

    def _information_gain(self, parent, left, right):
        parent_entropy = self._entropy(parent)
        left_entropy = self._entropy(left)
        right_entropy = self._entropy(right)

        weighted_avg_entropy = (len(left) / len(parent)) * left_entropy + (len(right) / len(parent)) * right_entropy
        return parent_entropy - weighted_avg_entropy

    def _entropy(self, y):
        class_probs = np.bincount(y) / len(y)
        return -np.sum(class_probs * np.log2(class_probs  + 1e-9))

    def predict(self, X):
        return [self._predict_single(x, self.tree) for x in X]

    def _predict_single(self, x, tree):
        if 'class' in tree:
            return tree['class']

        feature_val = x[tree['feature_idx']]
        if feature_val <= tree['threshold']:
            return self._predict_single(x, tree['left_tree'])
        else:
            return self._predict_single(x, tree['right_tree'])

In [3]:
# Step-2 : Loading and Splitting the Iris Dataset : 
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

data = load_iris()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [4]:
# Step-3 : Training the Custom Decision Tree :
custom_tree = CustomDecisionTree(max_depth = 3)
custom_tree.fit(X_train, y_train)

y_pred_custom = custom_tree.predict(X_test)

accuracy_custom = accuracy_score(y_test, y_pred_custom)
print(f"Custom Decision Tree Accuracy: {accuracy_custom:.4f}")

Custom Decision Tree Accuracy: 0.8000


In [5]:
# Step-4 : Train the Scikit-learn decision tree :
sklearn_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
sklearn_tree.fit(X_train, y_train)

y_pred_sklearn = sklearn_tree.predict(X_test)

accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)
print(f"Scikit-learn Decision Tree Accuracy: {accuracy_sklearn:.4f}")

Scikit-learn Decision Tree Accuracy: 1.0000


In [6]:
# Step-5 : Result Comparison :
print(f"Accuracy Comparison:")
print(f"Custom Decision Tree: {accuracy_custom:.4f}")
print(f"Scikit-learn Decision Tree: {accuracy_sklearn:.4f}")

Accuracy Comparison:
Custom Decision Tree: 0.8000
Scikit-learn Decision Tree: 1.0000


In [7]:
# Task-3.1 : Implementing Classification Models :
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import f1_score, mean_squared_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [16]:
# Load the Wine dataset
data = load_wine()
X = data.data   # Features
y = data.target # Labels

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shape of training data:", X_train.shape)
print("Shape of test data:", X_test.shape)


Shape of training data: (142, 13)
Shape of test data: (36, 13)


In [17]:
# Create and train a Decision Tree Classifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

# Predict on test data
y_pred_dt = dt.predict(X_test)

# Calculate F1 score
f1_dt = f1_score(y_test, y_pred_dt, average='weighted')
print("F1 Score for Decision Tree:", f1_dt)


F1 Score for Decision Tree: 0.9439974457215836


In [18]:
# Create and train a Random Forest Classifier
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

# Predict on test data
y_pred_rf = rf.predict(X_test)

# Calculate F1 score
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')
print("F1 Score for Random Forest:", f1_rf)


F1 Score for Random Forest: 1.0


In [19]:
print(f"Decision Tree F1 Score: {f1_dt:.4f}")
print(f"Random Forest F1 Score: {f1_rf:.4f}")

Decision Tree F1 Score: 0.9440
Random Forest F1 Score: 1.0000


In [20]:
# Create a dictionary of hyperparameters to try
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10]
}

# Create the GridSearchCV object
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, n_jobs=-1, verbose=1)

# Fit the model
grid.fit(X_train, y_train)

# Best parameters
print("Best parameters:", grid.best_params_)

# Predict using the best model
y_pred_best_rf = grid.best_estimator_.predict(X_test)

# F1 score
f1_best_rf = f1_score(y_test, y_pred_best_rf, average='weighted')
print("F1 Score for Tuned Random Forest:", f1_best_rf)


Fitting 3 folds for each of 27 candidates, totalling 81 fits
Best parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
F1 Score for Tuned Random Forest: 1.0


In [21]:
# Create and train a Decision Tree Regressor
dt_reg = DecisionTreeRegressor(random_state=42)
dt_reg.fit(X_train, y_train)

# Predict on test data
y_pred_dt_reg = dt_reg.predict(X_test)

# Evaluate with Mean Squared Error (MSE)
mse_dt = mean_squared_error(y_test, y_pred_dt_reg)
print("MSE for Decision Tree Regressor:", mse_dt)

MSE for Decision Tree Regressor: 0.16666666666666666


In [22]:
# Create and train a Random Forest Regressor
rf_reg = RandomForestRegressor(random_state=42)
rf_reg.fit(X_train, y_train)

# Predict on test data
y_pred_rf_reg = rf_reg.predict(X_test)

# Evaluate MSE
mse_rf = mean_squared_error(y_test, y_pred_rf_reg)
print("MSE for Random Forest Regressor:", mse_rf)


MSE for Random Forest Regressor: 0.06483333333333333


In [23]:
from scipy.stats import randint

# Hyperparameters to try
param_dist = {
    'n_estimators': randint(50, 200),
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': randint(2, 10)
}

# RandomizedSearchCV
random_search = RandomizedSearchCV(RandomForestRegressor(random_state=42),
                                   param_distributions=param_dist,
                                   n_iter=10, cv=3, random_state=42, verbose=1, n_jobs=-1)

# Fit the model
random_search.fit(X_train, y_train)

# Best parameters
print("Best parameters (Regressor):", random_search.best_params_)

# Predict using best model
y_pred_best_rf_reg = random_search.best_estimator_.predict(X_test)

# Evaluate MSE
mse_best_rf_reg = mean_squared_error(y_test, y_pred_best_rf_reg)
print("MSE for Tuned Random Forest Regressor:", mse_best_rf_reg)


Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters (Regressor): {'max_depth': 5, 'min_samples_split': 4, 'n_estimators': 124}
MSE for Tuned Random Forest Regressor: 0.06332265278997057
